# IMPORTANTE:

**USE AS SEGUINTES VERSÕES PRAS BIBLIOTECAS:**

```
numpy 2.5.2
lightgbm 3.3.5
miceforest 5.7.0
```

In [13]:
import pandas as pd 
from pathlib import Path
import miceforest as mf
import numpy as np
import pandas as pd
import re
# from ydata_profiling import ProfileReport

# Direct attribute injections into NumPy for compatibility with miceforest,
# which uses an older version of it
np.NaN = np.nan
np.Inf = np.inf

# 1. NumPy 2.0 np.array(..., copy=False) compatibility patch
if not getattr(np.array, "_is_patched", False): # _is_patched guarantees that the patch is applied only once, even if this code is executed multiple times
    _orig_np_array = np.array

    def _safe_np_array(object, *args, **kwargs):
        if kwargs.get("copy") is False:
            try:
                return _orig_np_array(object, *args, **kwargs)
            except ValueError:
                kwargs.pop("copy", None)
                return np.asarray(object, *args, **kwargs)
        return _orig_np_array(object, *args, **kwargs)

    _safe_np_array._is_patched = True
    np.array = _safe_np_array


np.array = _safe_np_array

df = pd.read_csv('clinical_data.tsv', sep='\t')

C:\Users\edubi\AppData\Local\Temp\ipykernel_21800\3516847396.py:33: DtypeWarning: Columns (0: MedR Adjuvant Radiation, 1: Age at Diagnosis, 2: MedR Age at Diagnosis, 3: American Joint Committee on Cancer Metastasis Stage Code, 4: Neoplasm Disease Lymph Node Stage American Joint Committee on Cancer Code, 5: Neoplasm Disease Stage American Joint Committee on Cancer Code, 6: American Joint Committee on Cancer Publication Version Type, 7: American Joint Committee on Cancer Tumor Stage Code, 8: Alpelisib Scheduling, 9: Angiolymphatic Invasion, 10: PRD Ever Cancer Location Ascites, 11: PRD Ever Cancer Location Axillary Lymph Node, 12: PRD Ever Cancer Location Bone, 13: PRD Ever Cancer Location Brain, 14: PRD Ever Cancer Location Breast, 15: PRD Ever Cancer Location Chest Wall, 16: PRD Ever Cancer Location Liver, 17: PRD Ever Cancer Location Lung, 18: PRD Ever Cancer Location Other Lymph Node, 19: PRD Ever Cancer Location Ovary, 20: PRD Ever Cancer Location Pleural Effusion, 21: PRD Ever Canc

## Aggregating CBio Studies

In [14]:
def get_studies_with_repeated_patients(
    directory_path="studies", threshold=5, column_name="Patient ID"
):
    studies_dir = Path(directory_path)
    matching_files = []

    for file_path in studies_dir.glob("*.tsv"):
        df = pd.read_csv(file_path, sep="\t", usecols=[column_name])

        if (df[column_name].value_counts() >= threshold).any():
            matching_files.append(file_path.name)

    return matching_files

In [15]:
matching_studies5 = get_studies_with_repeated_patients(threshold = 5)
print(f"{len(matching_studies5)} studies with at least one patient repeated 5+ times:")
print(matching_studies5)

matching_studies3 = get_studies_with_repeated_patients(threshold = 3)
print(f"\n{len(matching_studies3)} studies with at least one patient repeated 3+ times:")
print(matching_studies3)

7 studies with at least one patient repeated 5+ times:
['brca_bccrc_xenograft_2014_clinical_data.tsv', 'brca_hta9_htan_2022_clinical_data.tsv', 'brca_mbcproject_2022_clinical_data.tsv', 'brca_mbcproject_wagle_2017_clinical_data.tsv', 'breast_alpelisib_2020_clinical_data.tsv', 'breast_ink4_msk_2021_clinical_data.tsv', 'breast_msk_2025_clinical_data.tsv']

10 studies with at least one patient repeated 3+ times:
['brca_bccrc_xenograft_2014_clinical_data.tsv', 'brca_hta9_htan_2022_clinical_data.tsv', 'brca_jup_msk_2020_clinical_data.tsv', 'brca_mbcproject_2022_clinical_data.tsv', 'brca_mbcproject_wagle_2017_clinical_data.tsv', 'brca_pareja_msk_2020_clinical_data.tsv', 'breast_alpelisib_2020_clinical_data.tsv', 'breast_ink4_msk_2021_clinical_data.tsv', 'breast_msk_2018_clinical_data.tsv', 'breast_msk_2025_clinical_data.tsv']


In [16]:
longitudinal_studies = ['breast_alpelisib_2020_clinical_data.tsv',
                       'brca_mbcproject_2022_clinical_data.tsv',
                       'brca_mbcproject_wagle_2017_clinical_data.tsv',]

In [17]:
def commonize_columns (df):
    rename_dict = {
        # Whitespace cleanup targets
        "PATH HER2 IHC ": "PATH HER2 IHC",
        "PATH Procedure Type ": "PATH Procedure Type",
        "PATH HER2 Status ": "PATH HER2 Status",
        "PATH HER2 Copy ": "PATH HER2 Copy",
        "PRD Hispanic ": "PRD Hispanic",
        # Demographics standardization
        "MedR Sex": "Sex",
        "Diagnosis Age": "Age at Diagnosis",
        "MedR Age at Diagnosis": "Age at Diagnosis",
        "PRD Race": "Race",
        "Race Category": "Race",
        "Ethnicity Category": "Ethnicity",
        "PRD Hispanic": "Ethnicity",
        # Specific duplicates
        "Cancer Type Detailed.1": "Cancer Type Detailed",
        "Weeks on Study": "Timepoint",
        "Sample Timepoint": "Timepoint",
    }

    df = df.rename(columns=lambda x: x.strip()).rename(columns=rename_dict)

    if df.columns.has_duplicates:
        df = df.loc[:, ~df.columns.duplicated(keep="first")]

    return df

In [18]:
def get_common_columns(file_names, directory_path="studies", commonize=False):
    """Finds columns that are present across all specified TSV files.

    Returns an empty set if file_names is empty or files are unreadable.
    """
    studies_dir = Path(directory_path)
    common_cols = None

    for name in file_names:
        file_path = studies_dir / name
        try:
            df = pd.read_csv(file_path, sep="\t", nrows=0)

            if commonize:
                df = commonize_columns(df)
                
            cols = set(df.columns)

            if common_cols is None:
                common_cols = cols
            else:
                common_cols &= cols 

        except Exception as e:
            print(f"Error reading header from {name}: {e}")

    return list(common_cols) if common_cols is not None else []

print(get_common_columns(longitudinal_studies, commonize=True))

['Cancer Type Detailed', 'Mutation Count', 'Fraction Genome Altered', 'TMB (nonsynonymous)', 'Cancer Type', 'Patient ID', 'Sex', 'Study ID', 'Timepoint', 'Sample ID', 'Number of Samples Per Patient']


In [19]:
def concatenate_dataframes(file_list, directory_path="studies"):
    studies_dir = Path(directory_path)
    processed_dfs = []

    for file_name in file_list:
        file_path = studies_dir / file_name

        df = pd.read_csv(file_path, sep="\t", low_memory=False)
        df_cleaned = commonize_columns(df)

        df_cleaned["Source_Dataset"] = file_name

        processed_dfs.append(df_cleaned)
        print(
            f"Successfully processed {file_name} ({len(df_cleaned)} rows)"
        )

    combined_df = pd.concat(processed_dfs, ignore_index=True, join="outer")
    return combined_df

In [20]:
df_cbio_concatenated = concatenate_dataframes(longitudinal_studies)
df_cbio_concatenated.to_csv("df_cbio_raw.csv", index=False)

Successfully processed breast_alpelisib_2020_clinical_data.tsv (141 rows)
Successfully processed brca_mbcproject_2022_clinical_data.tsv (379 rows)
Successfully processed brca_mbcproject_wagle_2017_clinical_data.tsv (237 rows)


In [21]:
def get_mostly_null_columns(df: pd.DataFrame, threshold: float = 0.5) -> list:
    null_ratios = df.isnull().mean()
    mostly_null_cols = null_ratios[null_ratios > threshold].index.tolist()

    return mostly_null_cols

In [22]:
mostly_null = get_mostly_null_columns(df_cbio_concatenated, threshold=0.5)
df_cbio_pruned = df_cbio_concatenated.drop(columns=mostly_null)

df_cbio_pruned = df_cbio_pruned.sort_values(
        by=['Source_Dataset', 'Patient ID', 'Timepoint'],
        ascending=[True, True, True],
        na_position="last",
    ).reset_index(drop=True)

In [23]:
df_cbio_pruned.to_csv('datasets/df_cbio_pruned.csv', index=False)

In [24]:
is_repeated = df_cbio_pruned.duplicated(subset=['Source_Dataset', 'Patient ID'], keep=False)

df_cbio_long = df_cbio_pruned[is_repeated].copy().reset_index(drop=True)
df_cbio_long.to_csv('datasets/df_cbio_pruned_longitudinal.csv', index=False)

## Setting up GENIE Dataset

In [25]:
df_genie = pd.read_csv('genie_data.tsv', sep='\t')

In [26]:
df_genie_long = df_genie[df_genie['Number of Samples Per Patient'] > 1]
df_genie_long = df_genie_long.drop(columns=['Sample Type', 'Histology.1']).rename(columns={'Sample Type.1': 'Sample Type'})

In [37]:
df_genie_long.to_csv("datasets/df_genie_long.csv", index=False)

## Concatenating CBio and GENIE

In [28]:
def standardize_cbio_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Renomeia as colunas do CBio para a nomenclatura padrão do GENIE.
    Adiciona o sufixo ' (MedR)' às colunas de prontuário clínico que colidem 
    com dados de laudo patológico (PATH).
    
    Parâmetros:
        df (pd.DataFrame): DataFrame com as colunas originais do CBio.
        
    Retorna:
        pd.DataFrame: DataFrame com as colunas renomeadas sem colisões de nomes.
    """
    rename_mapping = {
        # --- Idade e Metadados ---
        "Age at Diagnosis": "Age at initial diagnosis (years)",
        "PATH Procedure Location": "Site of Sample tested",
        "MedR Time to Metastatic Diagnosis (Calculated Months)": "Time from initial diagnosis to metastatic diagnosis (months)",
        "MedR Stage at Diagnosis": "Stage at Diagnosis",

        # --- Grau Tumoral (PATH vs MedR) ---
        "PATH Sample Grade": "Grade",
        "MedR Diagnostic Grade": "Grade (MedR)",

        # --- Histologia e Lateralidade (PATH vs MedR) ---
        "PATH Sample Histology": "Histology",
        "MedR Diagnostic Histology": "Histology (MedR)",
        "PATH Breast Procedure Laterality": "Tumor Registry Laterality",
        "MedR Diagnosis Laterality": "Tumor Registry Laterality (MedR)",

        # --- Receptores e Biomarcadores (PATH vs MedR) ---
        "PATH Estrogen Receptor Status": "ER Summary Status",
        "MedR Diagnostic ER Status": "ER Summary Status (MedR)",
        
        "PATH Progesterone Receptor Status": "PR Summary Status",
        "MedR Diagnostic PR Status": "PR Summary Status (MedR)",
        
        "PATH HER2 Status": "HER2 Summary Result",
        "MedR Diagnostic HER2 Status": "HER2 Summary Result (MedR)",
        
        "PATH HER2 IHC": "HER2 IHC Lab Interpretation at Diagnosis",
        "MedR Diagnostic HER2 IHC": "HER2 IHC Lab Interpretation at Diagnosis (MedR)",

        # --- Sítios Metastáticos: Histórico Geral (Ever) vs Momento do Diagnóstico Metastático (at Mets Dx) ---
        "MedR Ever Bone Mets": "Distant Mets: Bone",
        "MedR Bone Mets at Mets Dx": "Distant Mets: Bone (Mets Dx)",
        
        "MedR Ever Brain/CNS Mets": "Distant Mets: Brain",
        "MedR Brain/CNS Mets at Mets Dx": "Distant Mets: Brain (Mets Dx)",
        
        "MedR Ever Liver Mets": "Distant Mets: Liver",
        "MedR Liver Mets at Mets Dx": "Distant Mets: Liver (Mets Dx)",
        
        "MedR Ever Lung Mets": "Distant Mets: Lung",
        "MedR Lung Mets at Mets Dx": "Distant Mets: Lung (Mets Dx)",
        
        "MedR Ever Adrenal Gland Mets": "Distant Mets: Adrenal",
        "MedR Adrenal Gland Mets at Mets Dx": "Distant Mets: Adrenal (Mets Dx)",
        
        "MedR Ever Distant Lymph Node Mets": "Distant Mets: Lymph Nodes",
        "MedR Distant Lymph Node Mets at Mets Dx": "Distant Mets: Lymph Nodes (Mets Dx)",
        
        "MedR Ever Pleural Effusion Mets": "Distant Mets: Pleura",
        "MedR Pleural Effusion Mets at Mets Dx": "Distant Mets: Pleura (Mets Dx)",
        
        "MedR Ever Skin Mets": "Distant Mets: Subcutaneous Tissue",
        "MedR Skin Mets at Mets Dx": "Distant Mets: Subcutaneous Tissue (Mets Dx)",
        
        "MedR Ever Soft Tissue Mets": "Distant Mets: Soft Tissue",
        "MedR Soft Tissue Mets at Mets Dx": "Distant Mets: Soft Tissue (Mets Dx)",
    }
    
    return df.rename(columns=rename_mapping)

In [29]:
df_cbio_long = standardize_cbio_columns(df_cbio_long)

df_cbio_long['Source Repository'] = 'cBioPortal'
df_genie_long['Source Repository'] = 'GENIE'

In [30]:
df_concat = pd.concat([df_cbio_long, df_genie_long], ignore_index=True, sort=False)

## Treating the concatenated dataset

In [31]:
def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Standardizes all DataFrame column names:

    - Converts all text to lowercase
    - Replaces spaces, punctuation, symbols, and special characters with
    underscores
    - Collapses multiple consecutive underscores into a single underscore
    - Removes leading and trailing underscores
    """

    def clean_column_name(col: str) -> str:
        # Convert to lowercase and remove surrounding whitespace
        clean_str = str(col).lower().strip()
        # Replace all non-alphanumeric characters with underscores
        clean_str = re.sub(r"[^a-z0-9]+", "_", clean_str)
        # Remove any leading or trailing underscores
        clean_str = clean_str.strip("_")
        return clean_str

    df_standardized = df.copy()
    df_standardized.columns = [
        clean_column_name(c) for c in df_standardized.columns
    ]
    return df_standardized


def fuse_all_duplicate_columns(
    df: pd.DataFrame, random_state: int = None
) -> pd.DataFrame:
    """Finds all duplicated column names in a DataFrame and fuses each pair into a

    single column using non-null preference and random selection for ties.
    """

    def fuse_two_series(
        s1: pd.Series, s2: pd.Series, random_state: int = None
    ) -> pd.Series:
        """Fuses two pandas Series:

        - If only one has a non-null value, picks that value.
        - If both are non-null, picks one at random (50/50 chance).
        - If both are null, returns NaN.
        """
        s1_valid = s1.notna()
        s2_valid = s2.notna()

        # Start with all NaNs
        result = pd.Series(index=s1.index, dtype=object)

        # 1. Only s1 is valid
        result[s1_valid & ~s2_valid] = s1[s1_valid & ~s2_valid]

        # 2. Only s2 is valid
        result[~s1_valid & s2_valid] = s2[~s1_valid & s2_valid]

        # 3. Both are valid -> pick randomly
        both_valid = s1_valid & s2_valid
        if both_valid.any():
            rng = np.random.default_rng(random_state)
            # Random boolean mask: True -> choose s1, False -> choose s2
            pick_s1 = rng.choice([True, False], size=both_valid.sum())
            chosen = np.where(
                pick_s1, s1.loc[both_valid].values, s2.loc[both_valid].values
            )
            result.loc[both_valid] = chosen

        return result

    df_fused = pd.DataFrame(index=df.index)
    unique_cols = pd.Series(df.columns).unique()

    for col in unique_cols:
        col_data = df[col]
        # If the column appears more than once, it will be returned as a DataFrame
        if isinstance(col_data, pd.DataFrame):
            if col_data.shape[1] == 2:
                df_fused[col] = fuse_two_series(
                    col_data.iloc[:, 0],
                    col_data.iloc[:, 1],
                    random_state=random_state,
                )
            else:
                raise ValueError(
                    f"Column '{col}' appears {col_data.shape[1]} times. Expected 2."
                )
        else:
            df_fused[col] = col_data

    return df_fused


def clean_and_extract_timepoint(
    df: pd.DataFrame, col_name: str = "timepoint"
) -> pd.DataFrame:
    df = df.copy()

    # Step 1: Replace values ending with 'BLOOD_P' with the next row's value
    is_blood_p = df[col_name].astype(str).str.endswith("BLOOD_P")
    df.loc[is_blood_p, col_name] = df[col_name].shift(-1)[is_blood_p]

    # Step 2: Extract substring after the last underscore, then extract only the digits
    extracted = (
        df[col_name]
        .astype(str)
        .str.extract(r"_([^_]+)$", expand=False)
        .str.extract(r"(\d+)", expand=False)
    )

    # Step 3: Convert to numeric nullable integer
    df[col_name] = pd.to_numeric(extracted, errors="coerce").astype(float)

    return df


def clean_values(df: pd.DataFrame) -> pd.DataFrame:
    # substitute empty values for nan
    df = df.replace(
        ["PARTICIPANT_DID_NOT_SUBMIT_SURVEY", "QUESTION_LEFT_BLANK", "NOT_FOUND_IN_RECORD", 
         "NOT_DONE", "UNKNOWN", "N/A_BLOOD_SAMPLE", "TESTING_PERFORMED_ON_DIFFERENT_SAMPLE", 
         "DON'T_KNOW", "Unknown or no information", "Not documented in medical record", "Test not done"],
        np.nan,
    )

    # get only number for timepoints
    df = clean_and_extract_timepoint(df, col_name="timepoint")

    column_mappings = {
        "cancer_type_detailed" : {"Invasive Breast Carcinoma" : "Breast Invasive Cancer, NOS"},
        "sex" : {"Female" : "FEMALE"}
    }

    df = df.replace(column_mappings)
    
    return df

def drop_high_null_columns(df: pd.DataFrame, threshold: float = 0.5) -> pd.DataFrame:
    null_proportions = df.isnull().mean()
    cols_to_drop = null_proportions[null_proportions >= threshold]

    if not cols_to_drop.empty:
        print(f"Columns with >= {threshold * 100:.0f}% null values:")
        for col, prop in cols_to_drop.items():
            print(f" - {col}: {prop * 100:.2f}% null")
    else:
        print(f"No columns found with >= {threshold * 100:.0f}% null values.")

    return df.drop(columns=cols_to_drop.index)


def impute_with_miceforest(
    df: pd.DataFrame, num_datasets: int = 1, iterations: int = 5, seed: int = 42
) -> pd.DataFrame:

    # miceforest handles categorical and numerical columns, but string columns
    # should be converted to 'category' dtype if not already done.
    df_impute = df.copy()

    # Re-categorize all categorical columns cleanly before passing to miceforest
    for col in df_impute.select_dtypes(include=["category", "object"]).columns:
        df_impute[col] = df_impute[col].astype(str).replace("nan", np.nan).astype("category")

    for col in df_impute.select_dtypes(include=["object"]).columns:
        df_impute[col] = df_impute[col].astype("category")

    # Create MICE Imputation Kernel
    kernel = mf.ImputationKernel(
        df_impute,
        save_all_iterations=False,
        random_state=seed,
    )

    # Run the MICE algorithm
    kernel.mice(iterations)

    # Return the first completed dataset
    completed_df = kernel.complete_data(dataset=0)
    return completed_df


def generate_ydata_profile(
    df: pd.DataFrame,
    output_html_path: str = "data_profile.html",
    title: str = "Data Profiling Report",
) -> ProfileReport:
    """Generates an exploratory data analysis profiling report using ydata-profiling

    and saves it to an HTML file.
    """
    profile = ProfileReport(df, title=title, explorative=True)
    profile.to_file(output_html_path)
    print(f"Profile report saved to: {output_html_path}")
    return profile

In [32]:
df_concat = standardize_columns(df_concat)
df_concat = fuse_all_duplicate_columns(df_concat)
df_concat = clean_values(df_concat)
df_concat_pruned = drop_high_null_columns(df_concat, threshold=0.75)

Columns with >= 75% null values:
 - prd_ever_cancer_location_ascites: 86.52% null
 - prd_ever_cancer_location_axillary_lymph_node: 86.52% null
 - prd_ever_cancer_location_bone: 86.52% null
 - prd_ever_cancer_location_brain: 86.52% null
 - prd_ever_cancer_location_breast: 86.52% null
 - prd_ever_cancer_location_chest_wall: 86.52% null
 - prd_ever_cancer_location_liver: 86.52% null
 - prd_ever_cancer_location_lung: 86.52% null
 - prd_ever_cancer_location_other_lymph_node: 86.52% null
 - prd_ever_cancer_location_ovary: 86.52% null
 - prd_ever_cancer_location_pleural_effusion: 86.52% null
 - prd_ever_cancer_location_skin: 86.52% null
 - path_cep17_copy: 92.88% null
 - path_her2_copy: 92.13% null
 - path_her2_cep17_ratio: 89.14% null
 - path_her2_fish: 87.45% null
 - path_ki67_percentage: 90.45% null
 - path_estrogen_receptor_percentage: 79.96% null
 - path_progesterone_receptor_percentage: 81.46% null
 - prd_histology_idc: 89.51% null
 - prd_histology_ilc: 89.51% null
 - prd_histology_midl

In [33]:
df_concat_imputed = impute_with_miceforest(df_concat_pruned, num_datasets=1, iterations=5)

C:\Users\edubi\AppData\Local\Temp\ipykernel_21800\1333952505.py:159: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_impute.select_dtypes(include=["category", "object"]).columns:
d:\reposGithub\processing-cbio-breast-cancer\.venv\Lib\site-packages\miceforest\ImputationKernel.py:370: UserWarning: [sample_id] have very rare categories, it is a good idea to group these, or set the min_data_in_leaf parameter to prevent lightgbm from outputting 0.0 probabilities.
  warn(


In [34]:
df_concat_pruned.to_csv('datasets/df_concat_pruned.csv', index=False)

In [35]:
df_concat_imputed.to_csv('datasets/df_concat_imputed.csv', index=False)

In [36]:
df_concat_imputed.describe()

,fraction_genome_altered,mutation_count,number_of_samples_per_patient,tmb_nonsynonymous,timepoint,path_sample_collection_time,medr_time_to_metastatic_diagnosis_calculated_days,time_from_initial_diagnosis_to_metastatic_diagnosis_months,age_at_sequencing,year_of_birth,...,number_of_pathology_reports,number_of_cancer_directed_drug_regimens_curated,number_of_ct_scans,number_of_mammographies_breast_only,number_of_mris,number_of_pet_or_pet_ct_scans,number_of_tumor_marker_ca15_3_results,number_of_tumor_marker_ca2729_results,number_of_tumor_marker_results,overall_survival_since_metastatic_diagnosis_months_used_to_make_the_km_survival_plot_in_the_cbioportal_instance
count,534.000000,534.000000,534.000000,534.000000,534.000000,534.000000,534.000000,534.000000,534.000000,534.000000,...,534.000000,534.000000,534.000000,534.000000,534.000000,534.000000,534.000000,534.000000,534.000000,534.000000
mean,0.300436,35.531835,2.700375,3.201345,1.953184,625.867041,1058.715356,34.382022,47.104869,1968.329588,...,10.749064,9.209738,12.900749,2.026217,7.294007,7.700375,8.243446,0.086142,8.421348,74.527400
std,0.208333,53.295821,0.906574,2.552861,0.895925,966.397666,1194.576189,39.565086,5.416279,5.251283,...,3.263412,3.493363,7.425394,1.764584,5.923661,5.359157,7.041601,0.985864,7.519106,27.141065
min,0.000000,0.000000,2.000000,0.000000,0.000000,-21.000000,0.000000,0.000000,29.000000,1957.000000,...,2.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,17.072368
25%,0.135075,3.000000,2.000000,1.541667,1.000000,26.000000,44.000000,1.000000,43.000000,1965.000000,...,10.000000,7.000000,8.000000,1.000000,4.000000,4.000000,3.000000,0.000000,3.000000,53.618421
50%,0.266700,7.000000,2.000000,2.566667,2.000000,170.000000,777.000000,23.000000,49.000000,1967.000000,...,11.000000,9.000000,13.000000,2.000000,6.000000,8.000000,8.000000,0.000000,8.000000,72.006579
75%,0.422525,49.750000,3.000000,4.400000,3.000000,930.000000,1940.000000,63.000000,50.000000,1971.000000,...,13.000000,12.000000,16.000000,2.000000,9.000000,11.500000,11.000000,0.000000,11.000000,82.532895
max,0.884600,342.000000,6.000000,19.574394,7.000000,7989.000000,5844.000000,192.000000,57.000000,1988.000000,...,22.000000,17.000000,44.000000,16.000000,45.000000,30.000000,37.000000,14.000000,38.000000,244.868421
